# demo_fix

> Phase 13, Track A (`specs/multiagent/phase13/`): fixes for Claude Haiku 5.5 teams. A1 (pooling on the clue board: `request`, `table`, the `evidence` instruction), A2 (one liar: `record`, `verify`, the `warn` instruction), A3 (the hidden-profile vote: `factsFirst`, `table`, the `dissent` instruction), each reported as the share of a gap closed (`FIX_PLAN`).

Artifacts (gitignored): `results/multiagent/fix/{quick,pilot,full}/` (clue board: `runs.csv`, `turns.jsonl`, `timing.jsonl`, `metrics.json`, `report.md` with every part, `fix.png`) and `results/multiagent/profile/{quick,pilot,full}/` (the vote: the same files, the report and figure for A3 only). The response cache and spend ledger are Phase 10's (`results/multiagent/llm/`), so one $150 API cap covers all live spend; the baseline cells equal Phase 11's, so seeds 900-904 replay its cached replies.

- **Quick** (normal tests): every arm with the fake client (which also asks and checks when the schema allows it); outputs identical for 1 and 8 processes (clue board: pooled and grid by grid; vote: 1 and 8 threads).
- **Pilot** and **main** (`--cell-timeout 0 --flags llm`, with `EXPLRL_LLM_LIVE=1`): live; the main runs only after `FIX_PLAN` is frozen and committed.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import time

import numpy as np

from explRL.bench_grid import readCSV, writeCSV
from explRL.clue_runs import runClueAny
from explRL.fix_grid import FIX_PLAN, fixGrid, fixPilotGrid, fixQuick, profileGrid, profilePilotGrid, profileQuick
from explRL.coop_grid import coopHash
from explRL.fix_report import figFix, fixLogMetrics, fixSummaries, writeFix
from explRL.llm_budget import PRICES, Budget
from explRL.llm_grid import HAIKU
from explRL.llm_runs import LLMConfig, makeClient
from explRL.llm_sched import mergeTiming, runWorkload, timingSummary
from explRL.profile_runs import runProfile
from explRL.train import demo_artifact_dir

FIX, VOTE = 'multiagent/fix', 'multiagent/profile'
WORKERS = min(8, os.cpu_count() or 1)
LLM_WORKERS, VOTE_WORKERS = 32, 16
CAP = 150.0
LLM = demo_artifact_dir(None, 'multiagent/llm')
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=CAP, ledger=str(LLM / 'spend.jsonl'))
P11 = demo_artifact_dir('full', 'multiagent/clue')


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def readLog(p):
    return [json.loads(x) for x in open(p)] if p.exists() else []


def collect(logs, out):
    "Concatenate the per-run turn logs (sorted by name) into turns.jsonl and remove them."
    with open(out / 'turns.jsonl', 'w') as f:
        for p in sorted(logs.glob('*.jsonl')) if logs.exists() else []:
            f.write(p.read_text())


def runFix(mode, grids, spec, workers):
    "The clue-board cells through runWorkload (runClueAny): runs.csv, turns.jsonl, timing.jsonl."
    out = demo_artifact_dir(mode, FIX)
    logs = out / 'turns'
    shutil.rmtree(logs, ignore_errors=True)
    t0 = time.time()
    res = runWorkload(grids, spec, workers, str(logs), run=runClueAny)
    wall = time.time() - t0
    writeCSV(res['rows'], out / 'runs.csv')
    collect(logs, out)
    res['timing'] = timingSummary(mergeTiming(logs, out / 'timing.jsonl'), wall)
    shutil.rmtree(logs, ignore_errors=True)
    return res, wall


def runVote(mode, cells, spec, workers):
    "The vote cells through runProfile (one client, batches of 100 episodes): runs.csv, turns.jsonl, timing.jsonl."
    out = demo_artifact_dir(mode, VOTE)
    logs = out / 'turns'
    shutil.rmtree(logs, ignore_errors=True)
    client = makeClient(LLMConfig.fromDict(cells[0]['cfg']), spec)
    t0 = time.time()
    rows = runProfile(cells, client, logs, workers=workers)
    wall = time.time() - t0
    writeCSV(rows, out / 'runs.csv')
    collect(logs, out)
    (out / 'timing.jsonl').write_text(''.join(json.dumps(x, sort_keys=True) + '\n' for x in client.timing))
    shutil.rmtree(logs, ignore_errors=True)
    return dict(rows=rows, timing=timingSummary(client.timing, wall), done=['vote'], refused=[], stopped=None), wall


def run(mode, grids, votes, spec, workers=None, voteWorkers=None, study='fixA', profileStudy='profileA'):
    "Run both workloads, then the report (report.md, fix.png, metrics.json) in both directories."
    live = spec.get('kind') == 'live'
    fres, fwall = runFix(mode, grids, spec, workers or (LLM_WORKERS if live else WORKERS))
    vres, vwall = runVote(mode, votes, spec, voteWorkers or (VOTE_WORKERS if live else WORKERS))
    rows, prows = fres['rows'], vres['rows']
    fo, vo = demo_artifact_dir(mode, FIX), demo_artifact_dir(mode, VOTE)
    p11 = readCSV(P11 / 'runs.csv') if (P11 / 'runs.csv').exists() else []
    S = fixSummaries(rows, p11, fixLogMetrics(readLog(fo / 'turns.jsonl')), prows, study, profileStudy)
    allRows = rows + prows
    meta = dict(mode=mode, planHash=coopHash(FIX_PLAN), commit=gitCommit(), model=HAIKU, cap=CAP, wall=fwall + vwall,
                pricesChecked=PRICES[HAIKU]['checked'], cost=float(sum(r['cost'] for r in allRows)),
                calls=int(sum(r['calls'] for r in allRows)), spent=Budget(CAP, spec['ledger']).spent() if spec.get('ledger') else 0.0)
    for out, res, parts in ((fo, fres, ('A1', 'A2', 'A3')), (vo, vres, ('A3',))):
        writeFix(out / 'report.md', S, meta, parts)
        if any(S[p] for p in parts):
            figFix(out, S, parts)
        (out / 'metrics.json').write_text(json.dumps(dict(meta, done=res['done'], refused=res['refused'], stopped=res['stopped'],
                                                          timing=res['timing'], plan=FIX_PLAN),
                                                     indent=2, sort_keys=True, default=str) + '\n')
        if live:                               # keep the live run's metrics and timing; a cached rerun overwrites the others
            for f in ('metrics.json', 'timing.jsonl'):
                if not (out / f.replace('.', '_live.')).exists():
                    shutil.copy(out / f, out / f.replace('.', '_live.'))
    print(f"{mode}: {len(rows)} board rows, {len(prows)} vote rows, {meta['calls']} calls, ${meta['cost']:.2f} for these "
          f"runs, ${meta['spent']:.2f} spent in all, {meta['wall']:.0f} s; stopped: {fres['stopped']}")
    return rows, prows, S, meta

## Quick: every arm and the vote with the fake client (pipeline check)

In [ ]:
t0 = time.time()
rows, prows, S, meta = run('quick', [('quick', fixQuick())], profileQuick(), dict(kind='fake'))
assert time.time() - t0 < 30
fq, vq = demo_artifact_dir('quick', FIX), demo_artifact_dir('quick', VOTE)
for d in (fq, vq):
    for f in ('runs.csv', 'turns.jsonl', 'timing.jsonl', 'metrics.json', 'report.md', 'fix.png'):
        assert (d / f).stat().st_size > 0, (d, f)
read = lambda: tuple((d / f).read_bytes() for d in (fq, vq) for f in ('runs.csv', 'turns.jsonl'))
ref = read()
for w, pooled in ((1, True), (8, False)):
    run('quick', [('quick', fixQuick())], profileQuick(), dict(kind='fake', pooled=pooled), workers=w, voteWorkers=w)
    assert read() == ref, w
# every part is reported; the new actions are used and logged (the fake client asks and checks when the schema allows)
assert set(S['A1']) == {f'{a}|{N}' for a in ('request', 'table', 'evidence') for N in (4, 8)}
assert set(S['A2']) == set(S['A2cost']) == {'record', 'verify', 'warn'} and len(S['A3']) == 6
m = fixLogMetrics(readLog(fq / 'turns.jsonl'))
by = lambda lv: [v for k, v in m.items() if k.split('|')[2] == lv]
assert sum(v['asks'] for v in by('request')) > 0 and sum(v['checks'] for v in by('verify') + by('verify|liar')) > 0
assert all(v['asks'] == v['checks'] == 0 for k, v in m.items() if k.split('|')[2] not in ('request', 'verify', 'verify|liar'))
recs = readLog(fq / 'turns.jsonl')
assert all('Requests:' in x['obs'] for x in recs if x['key'].split('|')[2] == 'request')
assert all('Known to the team:' in x['obs'] for x in recs if x['key'].split('|')[2] == 'table')
assert all('Requests:' not in x['obs'] and 'Known to the team:' not in x['obs'] and 'record ' not in x['obs']
           for x in recs if x['key'].split('|')[2] in ('baseline', 'baseline|liar'))
assert {r['level'] for r in prows} == {f'{a}|{v}' for a in ('baseline', 'factsFirst', 'table', 'dissent', 'solo') for v in ('classic', 'lone')}
for d in (fq, vq):
    for p in d.iterdir():
        if p.suffix in ('.csv', '.jsonl', '.json', '.md'):
            assert not re.search(r'sk-ant-|x-api-key|authorization|api[_-]?key', p.read_text(), re.I), p
print(open(fq / 'report.md').read()[:1500])

## Pilot (`--flags llm`, live, about 2 300 calls, about $0.55)

Each new arm at N = 4, seeds 900-901, 40 turns (A2 arms with the liar), so the main run replays these replies; the vote's `baseline` and `factsFirst` (classic) on 20 episodes and the solo ceiling on the same 20. AC-6: invalid replies under 5% for every new action and the vote; the vote's baseline win rate between 0.2 and 0.8 and the solo ceiling at least 0.8, else change `shared` and `unique` in `FIX_PLAN['profile']` and rerun the pilot once (T5).

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    prows_, vrows, _, pmeta = run('pilot', [('pilot', fixPilotGrid())], profilePilotGrid(), LIVE_SPEC, study='fixPilot',
                                  profileStudy='profilePilot')
    m = fixLogMetrics(readLog(demo_artifact_dir('pilot', FIX) / 'turns.jsonl'))
    for r in prows_:
        print(f"{r['level']:>12} seed {r['seed']}: invalid {r['invalid'] / max(r['calls'], 1):.3f}, welfare {r['welfare']:.2f}, "
              f"pooled share {r['pooledShare']:.2f}")
    for a, k in (('ask', 'asks'), ('check', 'checks')):
        bad, ok = sum(v[f'{a}Invalid'] for v in m.values()), sum(v[k] for v in m.values())
        print(f"'{a}': {ok} valid, {bad} invalid, invalid rate {bad / max(ok + bad, 1):.3f} (AC-6: < 0.05)")
    vlog = readLog(demo_artifact_dir('pilot', VOTE) / 'turns.jsonl')
    print(f"vote: invalid rate {np.mean([not x['valid'] for x in vlog]):.3f} (AC-6: < 0.05)")
    win = lambda lv: float(np.mean([r['success'] for r in vrows if r['level'] == lv]))
    print(f"vote: baseline win rate {win('baseline|classic'):.2f} (band 0.2-0.8), factsFirst {win('factsFirst|classic'):.2f}, "
          f"solo ceiling {win('solo|classic'):.2f} (at least 0.8)")

## Main (`--cell-timeout 0 --flags llm`, live, about 39 000 new calls, about $9)

`fixGrid` (A1 and A2, seeds 900-909, 40 turns; the baseline's seeds 900-904 replay Phase 11's cached replies) and `profileGrid` (A3, episode seeds 5000-5099, both variants, the solo ceiling); `FIX_PLAN` must be frozen and committed first (gate A).

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    assert FIX_PLAN['frozen'], 'freeze and commit FIX_PLAN before the main runs'
    rows, prows, S, meta = run('full', [('fixA', fixGrid())], profileGrid(), LIVE_SPEC)
    print(open(demo_artifact_dir('full', FIX) / 'report.md').read()[:4000])

## Cached rerun (`--flags llm`, after the main runs)

Every reply comes from the cache: `runs.csv` and `turns.jsonl` byte-identical, nothing spent (AC-7).

In [ ]:
#| llm
if not LIVE or not (demo_artifact_dir('full', FIX) / 'runs.csv').exists():
    print('skipped: needs EXPLRL_LLM_LIVE=1 and the main runs')
else:
    files = [demo_artifact_dir('full', d) / f for d in (FIX, VOTE) for f in ('runs.csv', 'turns.jsonl')]
    before, spent = [p.read_bytes() for p in files], Budget(CAP, LIVE_SPEC['ledger']).spent()
    run('full', [('fixA', fixGrid())], profileGrid(), LIVE_SPEC)
    assert [p.read_bytes() for p in files] == before and Budget(CAP, LIVE_SPEC['ledger']).spent() == spent
    print('cached rerun: byte-identical, free')